# 🟩 2층 신경망 역전파를 손으로 계산하기  

## 🟢 1. 입력과 약속  

입력 2개 → 은닉 2개 → 출력 1개. 행렬은 왼쪽에서 입력 벡터를 곱한다. `W1`의 한 행이 은닉 뉴런 하나에 해당한다. `@`는 행렬 곱셈, `*`는 같은 위치끼리의 곱셈이다.  

요구사항의 `y_true`가 비어 있으므로 결과 예시의 `y_true=1`을 사용한다. 빈 값을 원문에서 수정하지 않는다. Binary Cross-Entropy(BCE)는 이진 교차 엔트로피다. shape은 배열의 모양이다. NumPy 스칼라 shape은 `()`다.  

| 이름 | 값 | shape |  
| --- | --- | --- |
| x | [1, 0] | (2,) |  
| W1 | [[0.1, 0.2], [0.3, 0.4]] | (2,2) |  
| b1 | [0, 0] | (2,) |  
| W2 | [0.5, 0.6] | (2,) |  
| b2 | 0 | () |  
| y_true | 1 | () |  

<br><br>

## 🟢 2. 순전파 수식  

$$\sigma(z)=\frac{1}{1+e^{-z}},\quad \sigma'(z)=\sigma(z)(1-\sigma(z))$$  

$$z_1=W_1x+b_1\;(2,),\quad a_1=\sigma(z_1)\;(2,)$$  

$$z_2=W_2a_1+b_2\;(),\quad \hat y=\sigma(z_2)\;()$$  

$$L=-\left[y\log\hat y+(1-y)\log(1-\hat y)\right]\;()$$  

### 🟡 한 단계씩 손계산  

1. 첫 은닉 뉴런: $0.1\times1+0.2\times0+0=0.1$.  
2. 둘째 은닉 뉴런: $0.3\times1+0.4\times0+0=0.3$.  
3. $a_{1,1}=1/(1+e^{-0.1})=0.5249791875\ldots$.  
4. $a_{1,2}=1/(1+e^{-0.3})=0.5744425168\ldots$.  
5. $z_2=0.5\times0.5249791875+0.6\times0.5744425168=0.6071551038\ldots$.  
6. $\hat y=1/(1+e^{-0.6071551038})=0.6472915700\ldots$.  
7. $L=-\log(0.6472915700)=0.4349584369\ldots$.  

중간 계산에서는 충분한 자리수를 유지하고 최종 표시만 소수점 4자리로 반올림한다.  

| 체크포인트 | shape | 소수점 4자리 |  
| --- | --- | --- |
| z1 | (2,) | [0.1000, 0.3000] |  
| a1 | (2,) | [0.5250, 0.5744] |  
| z2 | () | 0.6072 |  
| y_pred | () | 0.6473 |  
| L | () | 0.4350 |  

In [1]:
from pathlib import Path  # 프로젝트 폴더를 찾는다.
import sys  # 모듈 검색 위치를 설정한다.
import numpy as np  # 수학 계산을 직접 수행한다.
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent  # 루트 또는 notebooks 폴더에서 실행한다.
assert (ROOT / "src").is_dir(), "Run from project root or notebooks folder."  # 실행 위치를 검사한다.
sys.path.insert(0, str(ROOT))  # 프로젝트 모듈을 불러올 수 있게 한다.
np.random.seed(42)  # 같은 난수로 재현한다.
np.set_printoptions(precision=10, suppress=True)  # 중간값을 충분한 자리수로 보여준다.

In [2]:
from src.backprop import backward, fixed_example, forward, gradient_check  # 직접 구현한 신경망을 가져온다.
example = fixed_example()  # 요구사항의 고정 숫자를 준비한다.
values, gradients = backward(**example)  # 순전파와 역전파를 직접 계산한다.
for name, value in values.items():  # 중간값을 하나씩 확인한다.
    print(name, "shape=", np.shape(value), "value=", value)  # 값과 shape을 함께 출력한다.

z1 shape= (2,) value= [0.1 0.3]
a1 shape= (2,) value= [0.5249791875 0.5744425168]
z2 shape= () value= 0.6071551038264654
y_pred shape= () value= 0.6472915699946178
loss shape= () value= 0.43495843685146063


## 🟢 3. 연쇄 법칙으로 출력부터 거꾸로 계산  

연쇄 법칙은 중간 계산의 미분을 곱해서 영향이 얼마나 전달되는지 구하는 규칙이다.  

### 🟡 출력 확률과 출력 선형값  

$$\frac{\partial L}{\partial\hat y}=-\frac{y}{\hat y}+\frac{1-y}{1-\hat y}\quad()$$  

$$\frac{\partial\hat y}{\partial z_2}=\hat y(1-\hat y)\quad()$$  

$$\frac{\partial L}{\partial z_2}=\left[-\frac{y}{\hat y}+\frac{1-y}{1-\hat y}\right]\hat y(1-\hat y)=-y(1-\hat y)+(1-y)\hat y=\hat y-y\quad()$$  

$y=1$이므로 $\partial L/\partial\hat y=-1/0.64729157=-1.5448988468\ldots$.  

첫 체크포인트: $\delta_2=\partial L/\partial z_2=0.64729157-1=-0.3527084300\ldots$.  

### 🟡 출력 가중치와 은닉 활성화  

$z_2=\sum_j W_{2,j}a_{1,j}+b_2$이므로 각 항을 미분한다.  

$$\frac{\partial L}{\partial W_2}=\delta_2a_1\quad(2,),\qquad\frac{\partial L}{\partial a_1}=\delta_2W_2\quad(2,)$$  

$$\frac{\partial L}{\partial W_2}=-0.3527084300\times[0.5249791875,0.5744425168]=[-0.1851645850,-0.2026107182]$$  

$$\frac{\partial L}{\partial a_1}=-0.3527084300\times[0.5,0.6]=[-0.1763542150,-0.2116250580]$$  

### 🟡 은닉 선형값과 첫 가중치  

$$\delta_1=\frac{\partial L}{\partial z_1}=\frac{\partial L}{\partial a_1}\odot a_1\odot(1-a_1)\quad(2,)$$  

$\odot$는 같은 위치끼리 곱하는 기호다.  

$$\sigma'(0.1)=0.2493760402\ldots,\quad\sigma'(0.3)=0.2444583117\ldots$$  

두 번째 체크포인트:  

$$\delta_1=[-0.1763542150\times0.2493760402, -0.2116250580\times0.2444583117]=[-0.0439785158,-0.0517335044]$$  

$z_{1,i}=\sum_j W_{1,ij}x_j+b_{1,i}$이므로 $\partial L/\partial W_{1,ij}=\delta_{1,i}x_j$다.  

$$\frac{\partial L}{\partial W_1}=\delta_1x^T=\operatorname{outer}(\delta_1,x)=\begin{bmatrix}-0.0439785158&0\\-0.0517335044&0\end{bmatrix}\quad(2,2)$$  

입력의 두 번째 값이 0이라 둘째 열의 기울기도 0이다. 추가로 편향의 기울기는 $\partial L/\partial b_1=\delta_1\;(2,)$, $\partial L/\partial b_2=\delta_2\;()$다.  

| 필수 미분 | shape | 손계산 소수점 4자리 |  
| --- | --- | --- |
| dL/dy_pred | () | -1.5449 |  
| dL/dz2 | () | -0.3527 |  
| dL/dW2 | (2,) | [-0.1852, -0.2026] |  
| dL/da1 | (2,) | [-0.1764, -0.2116] |  
| dL/dz1 | (2,) | [-0.0440, -0.0517] |  
| dL/dW1 | (2,2) | [[-0.0440, 0], [-0.0517, 0]] |  

원문 참고 예시의 `z2=0.6071`과 은닉 기울기 둘째 값 `-0.0513`은 정확한 반올림 결과와 다르다. 원문은 보존하고 위 식과 실제 계산을 기준으로 검증한다.  

In [3]:
manual_forward = {"z1": [0.1000, 0.3000], "a1": [0.5250, 0.5744], "z2": 0.6072, "y_pred": 0.6473, "loss": 0.4350}  # 손계산의 반올림 결과를 독립적으로 기록한다.
manual_gradients = {"y_pred": -1.5449, "z2": -0.3527, "W2": [-0.1852, -0.2026], "a1": [-0.1764, -0.2116], "z1": [-0.0440, -0.0517], "W1": [[-0.0440, 0], [-0.0517, 0]]}  # 필수 기울기 기준을 기록한다.
for name, expected in manual_forward.items():  # 모든 순전파 값을 검사한다.
    np.testing.assert_array_equal(np.round(values[name], 4), expected)  # 손계산과 소수점 4자리까지 비교한다.
for name, expected in manual_gradients.items():  # 모든 필수 역전파 값을 검사한다.
    np.testing.assert_array_equal(np.round(gradients[name], 4), expected)  # 손계산과 소수점 4자리까지 비교한다.
    print("dL/d" + name, "shape=", np.shape(gradients[name]), "value=", gradients[name])  # 정확한 값과 shape을 출력한다.
print("PASS: all handwritten checkpoints match to four decimal places")  # 실제 통과를 표시한다.

dL/dy_pred shape= () value= -1.5448988467566709
dL/dz2 shape= () value= -0.3527084300053822
dL/dW2 shape= (2,) value= [-0.185164585  -0.2026107182]
dL/da1 shape= (2,) value= [-0.176354215 -0.211625058]
dL/dz1 shape= (2,) value= [-0.0439785158 -0.0517335044]
dL/dW1 shape= (2, 2) value= [[-0.0439785158 -0.          ]
 [-0.0517335044 -0.          ]]
PASS: all handwritten checkpoints match to four decimal places


## 🟢 4. 독립적인 수치 미분으로 재검증  

손계산과 같은 식을 코드로 옮기면 같은 실수를 반복할 수 있다. 따라서 한 매개변수만 조금 바꾸어 손실이 얼마나 변하는지 직접 잰다.  

$$g_{numerical}=\frac{L(\theta+h)-L(\theta-h)}{2h},\quad h=10^{-5}$$  

모든 `W1`, `b1`, `W2`, `b2` 원소를 검사한다. 이 방법은 검증용이다. 매개변수가 많으면 계산이 비싸므로 실제 학습에는 역전파를 사용한다.  

In [4]:
errors = gradient_check(example)  # 모든 가중치와 편향의 중심차분을 계산한다.
print("Maximum absolute errors:", errors)  # 각 매개변수의 최대 차이를 출력한다.
assert max(errors.values()) < 1e-8  # 분석 기울기가 독립 검사와 일치하는지 확인한다.
updated = dict(example)  # 한 번 학습할 복사본을 만든다.
for name in ["W1", "b1", "W2", "b2"]:  # 학습 대상 네 종류를 고른다.
    updated[name] = example[name] - 0.1 * gradients[name]  # 기울기 반대 방향으로 이동한다.
new_loss = forward(**updated)["loss"]  # 이동 후 손실을 구한다.
print("Loss before:", values["loss"], "after:", new_loss)  # 실제 학습 효과를 비교한다.
assert new_loss < values["loss"]  # 손실이 줄었는지 확인한다.

Maximum absolute errors: {'W1': 3.695856121144203e-12, 'b1': 3.695856121144203e-12, 'W2': 3.2595592891482283e-12, 'b2': 1.6270873537393982e-12}
Loss before: 0.43495843685146063 after: 0.4144299462119412


## 🟢 5. 더 공부할 내용  

- 배치 학습에서는 샘플별 손실을 평균내고 기울기도 같은 기준으로 평균내야 한다.  
- 벡터의 미분을 모은 행렬은 Jacobian(야코비안)이다. 역전파는 전체 Jacobian을 만들지 않고 필요한 벡터와 곱한다.  
- Sigmoid가 0 또는 1에 가까우면 은닉층 미분이 작아지는 포화 현상이 생긴다.  
- BCE와 Sigmoid를 결합한 출력 기울기는 `p-y`로 간단해진다. 손실 자체는 `logaddexp(0,z)-y*z`로 계산하면 `log(0)` 문제를 피한다.  
- 다음 과제에서는 각 연산의 부모와 미분 규칙을 저장한 계산 그래프로 작은 자동 미분 엔진을 직접 만들 수 있다. 이번 과제에서는 자동 미분 라이브러리를 사용하지 않는다.  

설명 연습: “손실의 기울기를 출력에서 입력 방향으로 전달하고, 그 기울기의 반대 방향으로 가중치를 바꾸면 손실을 줄일 수 있다.”  